# 🔧 ID-VLM — Notebook 03: LoRA Fine-Tuning

**Goal:** Fine-tune Qwen2-VL-2B via LoRA using Unsloth's `FastVisionModel` on our MIDV-2020 instruction pairs.

This notebook uses the **exact same LoRA/Unsloth stack as FinSight AI** — same fine-tuning muscle, applied to vision.

---

## What this notebook does:
1. Install Unsloth + dependencies
2. Load Qwen2-VL-2B in 4-bit with LoRA adapters
3. Load the training data from Notebook 01
4. Fine-tune with `SFTTrainer` + `UnslothVisionDataCollator`
5. Save the fine-tuned model to Google Drive

**Expected runtime:** ~30-60 minutes on T4 for 300 steps.

## 1. Setup & Install

In [ ]:
%%capture
# Install Unsloth with vision support
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install qwen-vl-utils
!pip install python-Levenshtein

In [ ]:
# Mount Drive & setup paths
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import torch

PROJECT_DIR = '/content/drive/MyDrive/id-vlm'
REPO_DIR = '/content/id-vlm'
sys.path.insert(0, REPO_DIR)

# Verify prerequisites
assert os.path.exists(f'{PROJECT_DIR}/data/processed/train.jsonl'), \
    'Training data not found! Run Notebook 01 first.'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'✅ Setup complete')

## 2. Load Model + Attach LoRA

Key decisions:
- `finetune_vision_layers=True` — adapt the vision encoder for document images
- `finetune_language_layers=True` — adapt the language decoder for JSON output
- `r=16` — LoRA rank (good balance of capacity vs. memory)
- `load_in_4bit=True` — fits on free T4 (16GB VRAM)

In [ ]:
from unsloth import FastVisionModel

# Load the base model in 4-bit quantization
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",  # 70% less VRAM
)

print(f'\n✅ Base model loaded')
print(f'Model type: {type(model).__name__}')
print(f'Device: {model.device}')

In [ ]:
# Attach LoRA adapters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,      # ← Adapt vision encoder for document images
    finetune_language_layers=True,     # ← Adapt language decoder for JSON output
    finetune_attention_modules=True,   # ← Include attention layers
    finetune_mlp_modules=True,         # ← Include MLP layers
    r=16,                              # LoRA rank
    lora_alpha=16,                     # LoRA alpha (= r for stable training)
    lora_dropout=0,                    # No dropout (recommended by Unsloth)
    bias="none",                       # No bias terms
    random_state=42,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\n✅ LoRA adapters attached')
print(f'Trainable parameters: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'Total parameters:     {total:,}')

## 3. Load & Prepare Training Data

In [ ]:
from src.dataset import load_dataset
from PIL import Image

# Load training data
train_raw = load_dataset(f'{PROJECT_DIR}/data/processed/train.jsonl')
val_raw = load_dataset(f'{PROJECT_DIR}/data/processed/val.jsonl')

print(f'Training samples:   {len(train_raw)}')
print(f'Validation samples: {len(val_raw)}')

In [ ]:
# Convert JSONL data into the format Unsloth expects
# Each sample needs messages with PIL images (not paths)

import config

def prepare_sample_for_training(sample):
    """Convert a JSONL sample into Unsloth training format."""
    metadata = sample.get('metadata', {})
    image_path = metadata.get('image_path', '')
    
    # Load image
    try:
        image = Image.open(image_path).convert('RGB')
        
        # Resize if needed (Unsloth recommends 300-1000px)
        w, h = image.size
        max_dim = max(w, h)
        if max_dim > 1000:
            scale = 1000 / max_dim
            image = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    except Exception as e:
        print(f'  ⚠️ Could not load {image_path}: {e}')
        return None
    
    # Get the prompt and answer from messages
    user_msg = sample['messages'][0]
    assistant_msg = sample['messages'][1]
    
    # Extract text prompt
    prompt_text = ''
    for content in user_msg['content']:
        if content['type'] == 'text':
            prompt_text = content['text']
    
    answer_text = assistant_msg['content'][0]['text']
    
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': image},
                    {'type': 'text', 'text': prompt_text},
                ],
            },
            {
                'role': 'assistant',
                'content': [
                    {'type': 'text', 'text': answer_text},
                ],
            },
        ],
    }


# Convert all training samples
print('Preparing training data...')
train_dataset = []
skipped = 0

for i, sample in enumerate(train_raw):
    prepared = prepare_sample_for_training(sample)
    if prepared is not None:
        train_dataset.append(prepared)
    else:
        skipped += 1

print(f'\n✅ Prepared {len(train_dataset)} training samples (skipped {skipped})')

# Convert to HuggingFace Dataset
from datasets import Dataset
train_hf = Dataset.from_list(train_dataset)
print(f'HuggingFace Dataset: {train_hf}')

## 4. Fine-Tune with SFTTrainer

Using Unsloth's optimized training loop:
- `UnslothVisionDataCollator` — handles multimodal batching
- `remove_unused_columns=False` — preserves image data
- `batch_size=1` with `gradient_accumulation=4` — fits on T4

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Enable training mode
FastVisionModel.for_training(model)

# Configure training
training_args = SFTConfig(
    # Batch & accumulation
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,       # Effective batch = 4
    
    # Learning rate
    learning_rate=2e-4,
    warmup_steps=20,
    
    # Duration
    max_steps=300,                       # ~300 steps for a few hundred images
    # num_train_epochs=3,                # Alternative: use epochs instead of steps
    
    # Precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Logging
    logging_steps=1,
    logging_dir=f'{PROJECT_DIR}/outputs/logs',
    
    # Saving
    save_steps=50,
    save_total_limit=3,
    output_dir=f'{PROJECT_DIR}/outputs/checkpoints',
    
    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    
    # Required for Vision Fine-tuning
    remove_unused_columns=False,         # ← CRITICAL: keeps image data
    dataset_kwargs={"skip_prepare_dataset": True},  # ← CRITICAL: don't tokenize images
    
    # Seed
    seed=42,
    
    # Report to (optional)
    report_to="none",                    # Set to "wandb" if you want W&B tracking
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_hf,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    args=training_args,
)

print('✅ Trainer configured')
print(f'Max steps: {training_args.max_steps}')
print(f'Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'Learning rate: {training_args.learning_rate}')

In [ ]:
# ── TRAIN ──
# This takes ~30-60 minutes on a T4 for 300 steps

import time

print('🚀 Starting fine-tuning...')
print(f'Training {len(train_dataset)} samples for {training_args.max_steps} steps')
print()

start_time = time.time()

# Check GPU memory before training
gpu_mem_before = torch.cuda.memory_allocated() / 1e9
print(f'GPU memory before: {gpu_mem_before:.2f} GB')

# Train!
train_result = trainer.train()

elapsed = time.time() - start_time
gpu_mem_after = torch.cuda.max_memory_allocated() / 1e9

print(f'\n✅ Training complete!')
print(f'Total time:     {elapsed/60:.1f} minutes')
print(f'Peak GPU memory: {gpu_mem_after:.2f} GB')
print(f'Final loss:      {train_result.training_loss:.4f}')

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [entry['step'] for entry in log_history if 'loss' in entry]
losses = [entry['loss'] for entry in log_history if 'loss' in entry]

if steps:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, 'b-', alpha=0.3, label='Per-step loss')
    
    # Smoothed loss (rolling average)
    window = min(20, len(losses))
    if window > 1:
        import numpy as np
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
        plt.plot(steps[window-1:], smoothed, 'r-', linewidth=2, label=f'Smoothed (window={window})')
    
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('ID-VLM Fine-Tuning Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'{PROJECT_DIR}/outputs/training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {PROJECT_DIR}/outputs/training_loss.png')
else:
    print('No training logs available.')

## 5. Save Fine-Tuned Model

Three save options:
1. **LoRA adapter only** — small, loads on top of base model
2. **Merged model** — standalone, no adapter loading needed
3. **GGUF** — for local inference with llama.cpp

In [ ]:
# Save LoRA adapter (small, ~50-100MB)
lora_dir = f'{PROJECT_DIR}/outputs/checkpoints/lora_adapter'
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f'✅ LoRA adapter saved to {lora_dir}')

# Check size
import subprocess
result = subprocess.run(['du', '-sh', lora_dir], capture_output=True, text=True)
print(f'Size: {result.stdout.strip()}')

In [ ]:
# Save merged model (larger, ~2-3GB, standalone)
merged_dir = f'{PROJECT_DIR}/outputs/checkpoints/merged_model'

print('Merging LoRA weights into base model...')
model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method="merged_16bit",  # Full 16-bit merged weights
)
print(f'✅ Merged model saved to {merged_dir}')

result = subprocess.run(['du', '-sh', merged_dir], capture_output=True, text=True)
print(f'Size: {result.stdout.strip()}')

In [ ]:
# Save training metadata
training_metadata = {
    'model_base': 'Qwen2-VL-2B-Instruct',
    'fine_tuning_method': 'LoRA via Unsloth FastVisionModel',
    'lora_r': 16,
    'lora_alpha': 16,
    'finetune_vision_layers': True,
    'finetune_language_layers': True,
    'training_samples': len(train_dataset),
    'max_steps': training_args.max_steps,
    'learning_rate': training_args.learning_rate,
    'effective_batch_size': training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    'final_loss': train_result.training_loss,
    'training_time_minutes': elapsed / 60,
    'peak_gpu_memory_gb': gpu_mem_after,
    'gpu': torch.cuda.get_device_name(0),
    'dataset': 'MIDV-2020 (4 doc types subset)',
}

with open(f'{PROJECT_DIR}/outputs/training_metadata.json', 'w') as f:
    json.dump(training_metadata, f, indent=2)

print('✅ Training metadata saved')
for k, v in training_metadata.items():
    print(f'  {k}: {v}')

## 6. Quick Sanity Check

Run the fine-tuned model on a few samples to verify it works.

In [ ]:
# Quick inference test on 3 samples
FastVisionModel.for_inference(model)

from qwen_vl_utils import process_vision_info

val_samples = load_dataset(f'{PROJECT_DIR}/data/processed/val.jsonl')[:3]

print('Quick sanity check on validation samples:')
print('=' * 60)

for i, sample in enumerate(val_samples):
    metadata = sample.get('metadata', {})
    image_path = metadata.get('image_path', '')
    
    if not os.path.exists(image_path):
        print(f'  ⚠️ Image not found: {image_path}')
        continue
    
    image = Image.open(image_path).convert('RGB')
    
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': config.EXTRACTION_PROMPT},
        ],
    }]
    
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = tokenizer(text=[input_text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False)
    
    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    output_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # Get ground truth
    gt_text = sample['messages'][1]['content'][0]['text']
    
    print(f'\nSample {i+1} ({metadata.get("doc_type", "?")}):')
    print(f'  Ground truth: {gt_text[:100]}')
    print(f'  Prediction:   {output_text[:100]}')

print('\n✅ Sanity check complete. Proceed to Notebook 04 for full evaluation!')

In [ ]:
# Optional: Upload LoRA adapter to HuggingFace Hub
# Uncomment the following if you want to publish your model

# model.push_to_hub_merged(
#     "OmTilwar/ID-VLM-Qwen2-VL-2B-LoRA",
#     tokenizer,
#     save_method="lora",
#     token="your_hf_token_here",
# )

print('Training complete! Model saved to Google Drive.')
print(f'  LoRA adapter: {lora_dir}')
print(f'  Merged model: {merged_dir}')
print(f'  Metadata:     {PROJECT_DIR}/outputs/training_metadata.json')